In [1]:
import numpy as np
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import math
import torch.nn.functional as F
import torch.optim as optim

In [2]:
#Initiations

context_length = 256
vocab_size = 100256
embedding_dim = 128
batch_size = 4
num_heads = 4
head_dim = embedding_dim // num_heads
dropout = 0.2
n_layers = 5

In [3]:
with open('the-verdict.txt', 'r') as f:
    text = f.read()

In [4]:
print(type(text))
print(len(text))
print(text[:199])
print(text[-99:])

<class 'str'>
20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married 
it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


In [5]:
# #Sample tokenization
# sample = "Hello world! how are you doing today?"
encoding = tiktoken.get_encoding('cl100k_base')
# S_tokens = encoding.encode(sample)
# print(S_tokens)

# decodeding = encoding.decode(S_tokens)
# print(decodeding)

In [6]:
token_IDs = encoding.encode(text)
print(type(token_IDs))
print(f'Total tokens are: {len(token_IDs)}')
print(max(token_IDs))
print(token_IDs[:50])

<class 'list'>
Total tokens are: 4943
100242
[40, 473, 1846, 2744, 3463, 7762, 480, 285, 22464, 4856, 264, 12136, 35201, 313, 4636, 264, 1695, 12637, 3403, 313, 708, 433, 574, 912, 2294, 13051, 311, 757, 311, 6865, 430, 11, 304, 279, 2673, 315, 813, 27025, 11, 568, 1047, 12504, 813, 19354, 11, 12502, 264, 9257, 57896, 11]


In [7]:
decoded = encoding.decode(token_IDs)
print(decoded[:50])

I HAD always thought Jack Gisburn rather a cheap g


In [8]:
#Understanding the loop
for i in range(len(token_IDs)-context_length):
    input_ids = token_IDs[i:i+ context_length]
    target_ids = token_IDs[i+1: i+ context_length+1]
    # print(input_ids)
    # print(target_ids)

**Splitting the data into train and validation**

In [9]:
split_idx = int(0.9 * len(token_IDs))

train_tokens = token_IDs[:split_idx]
val_tokens = token_IDs[split_idx:]

print(f"Train tokens: {len(train_tokens)}")
print(f"Validation tokens: {len(val_tokens)}")

Train tokens: 4448
Validation tokens: 495


In [10]:
#creating input - target pairs
class LLMDataset(Dataset):
    def __init__(self, token_IDs, context_length):
        self.token_IDs = torch.tensor(token_IDs, dtype=torch.long) #Converting the input list into tensor
        self.context_length = context_length
    
    def __len__(self):
        return len(self.token_IDs) - self.context_length
        
    def __getitem__(self, idx):
        x = self.token_IDs[idx: idx + self.context_length] #Input tokens
        y = self.token_IDs[idx+1: idx + self.context_length+1] #Input tokens + 1

        return x, y

In [11]:
train_dataset = LLMDataset(train_tokens, context_length) 

val_dataset = LLMDataset(val_tokens, context_length)

In [12]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [13]:
# the very first row (index 0)
x, y = train_dataset[0]

print("--- Single Row Inspection ---")
print("Input Token IDs (X): ", x.tolist())
print("Target Token IDs (Y):", y.tolist())

--- Single Row Inspection ---
Input Token IDs (X):  [40, 473, 1846, 2744, 3463, 7762, 480, 285, 22464, 4856, 264, 12136, 35201, 313, 4636, 264, 1695, 12637, 3403, 313, 708, 433, 574, 912, 2294, 13051, 311, 757, 311, 6865, 430, 11, 304, 279, 2673, 315, 813, 27025, 11, 568, 1047, 12504, 813, 19354, 11, 12502, 264, 9257, 57896, 11, 323, 9749, 5678, 304, 264, 47625, 389, 279, 51768, 26919, 13, 320, 27831, 358, 4856, 3463, 433, 1053, 617, 1027, 22463, 477, 48606, 9456, 10227, 2673, 315, 813, 27025, 75857, 9210, 574, 1148, 279, 3278, 2663, 433, 13, 358, 649, 6865, 18083, 13, 480, 100242, 666, 24510, 313, 26301, 1566, 10780, 2503, 466, 313, 451, 501, 5620, 813, 653, 4711, 481, 671, 67, 20901, 13, 330, 2173, 3388, 433, 596, 2133, 311, 3708, 279, 907, 315, 856, 6945, 364, 3195, 709, 26, 719, 358, 1541, 956, 1781, 315, 430, 11, 4491, 13, 23194, 5721, 313, 1820, 4814, 311, 18925, 83, 374, 682, 358, 1781, 315, 1210, 578, 3492, 11, 389, 18083, 13, 666, 24510, 596, 23726, 11, 56016, 1202, 721, 5544,

In [14]:
#Verifying the dataloader
for x_batch, y_batch in train_dataloader:
    print(x_batch.shape)
    print(y_batch.shape)
    break

torch.Size([4, 256])
torch.Size([4, 256])


In [15]:
# embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
# S_token_embedding = embedding_layer(x_batch)
# # print(S_token_embedding)

# for x_batch, y_batch in dataloader:
#     result = embedding_layer(x_batch)
#     print(result.shape)
#     break

# pos_embedding_layer = nn.Embedding(context_length, embedding_dim)
# S_position_ids = torch.arange(context_length)
# posi_embedding = pos_embedding_layer(S_position_ids)
# print(posi_embedding.shape)

# S_Input_embedding = S_token_embedding+posi_embedding
# print(S_Input_embedding.shape)

In [16]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_length):
        super().__init__()
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.positional_embedding_layer = nn.Embedding(context_length, embedding_dim)

    def forward(self, x):
        token_embedding = self.embedding_layer(x) # Pass input tokens through the embedding layer to get semantic vectors
        position_ids = torch.arange(0, x.size(1), device=x.device) # Creating a sequence of position indices (0 to sequence_length - 1)
        pos_embedding = self.positional_embedding_layer(position_ids) #Passing postion ids to get position embeddings
        
        input_embedding = token_embedding + pos_embedding 
        return input_embedding

In [17]:
input_embeddings = InputEmbedding(vocab_size, embedding_dim, context_length)
IP_embedd_result = input_embeddings(x_batch)
# print(IP_embedd_result)

In [18]:
# query_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
# key_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
# value_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)

# input_embeddings = torch.randn(batch_size, context_length, embedding_dim)
# queries = query_layer(input_embeddings)
# keys = key_layer(input_embeddings)
# value = value_layer(input_embeddings)

# print(input_embeddings.shape)
# print(queries.shape)
# print(keys.shape)
# print(value.shape)


# K_transpose = keys.transpose(1, 2)
# attention_score = queries @ K_transpose
# scaled_attention_score = attention_score / math.sqrt(embedding_dim)
# # print(scaled_attention_score)
# print(scaled_attention_score.shape)

# attention_weights = F.softmax(scaled_attention_score, dim=1)
# # print(attention_weights)
# print(attention_weights.shape)

# context_vector = attention_weights @ value
# print(context_vector.shape)

In [19]:
class Selfattention(nn.Module):
    def __init__(self, embedding_dim, context_length):
        super().__init__()
        self.query_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.key_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.value_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.embedding_dim = embedding_dim
        self.mask = torch.tril(torch.ones(context_length, context_length))
        
    def forward(self, x):
        Q = self.query_layer(x)
        K = self.key_layer(x)
        V = self.value_layer(x)

        K_transpose = K.transpose(1, 2)
        attention_score = Q @ K_transpose

        masked_attn_scores = attention_score.masked_fill(self.mask==0, float('-inf'))

        scaled_attention_score = attention_score / math.sqrt(self.embedding_dim)
        attention_weights = F.softmax(scaled_attention_score, dim=2)
        
        context_vector = attention_weights @ V



        return context_vector

In [20]:
selfattention = Selfattention(embedding_dim, context_length)
result = selfattention(IP_embedd_result)
print(result.shape)

torch.Size([4, 256, 128])


In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self, num_heads, embedding_dim, context_length, dropout):
        super().__init__()
        assert (embedding_dim % num_heads == 0), "embedding_dim must be divisible by num_heads"
        
        self.q_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.k_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.v_layer = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(embedding_dim, embedding_dim)  # Linear layer to combine head outputs
        self.register_buffer("mask", torch.tril(torch.ones(context_length, context_length)))

    def forward(self, x):
        B, T, C = x.shape  #Extract B, T, C from the input shape

        Q = self.q_layer(x) #Projecting embeddings into raw tensors
        K = self.k_layer(x)
        V = self.v_layer(x)

        queries = Q.view(B, T, self.num_heads, self.head_dim) #Reshaping (B, T, C) --> (B, T, H, D) 
        keys = K.view(B, T, self.num_heads, self.head_dim)
        values = V.view(B, T, self.num_heads, self.head_dim)
        
        queries = queries.transpose(1, 2)  #Transforming (B, T, H, D) --> (B,H,T,D)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        

        K_transpose = keys.transpose(2, 3)  #(B, T, H, D) --> (B, T, D, H) 
        attention_score = queries @ K_transpose 
        mask = self.mask[:T, :T]  #Sclicing the predefined mask to adjust the input seq length
        masked_attn_scores = attention_score.masked_fill(mask==0, float('-inf')) #Filling with -inf where mask is 0
        scaled_attn_scores = masked_attn_scores / math.sqrt(self.head_dim) #Mean = 0, variance=1 or Scale by √head_dim
        attn_weights = F.softmax(scaled_attn_scores, dim=3) #probs sum up to 1
        attn_weights = self.dropout(attn_weights) #drop some attn wights

        context_vec = attn_weights @ values 
        context_vec = context_vec.transpose(1, 2) # (B, H, T, D) -> (B, T, H, D)
        context_vec = context_vec.contiguous().view(B, T, self.num_heads * self.head_dim) # (B, T, H, D) -> (B, T, H*D)
        context_vector = self.out_proj(context_vec) 


        return context_vector


In [22]:
mha = MultiheadAttention(num_heads, embedding_dim, context_length, 0.2)
Mha_result = mha(IP_embedd_result)
print(Mha_result.shape)

torch.Size([4, 256, 128])


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim), #(128, 4*128) 4x hidden layers
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim), #bring down the projected dimension
        )
    
    def forward(self, x):
        return self.layers(x)

In [24]:
ff = FeedForward(embedding_dim)
ff_result = ff(Mha_result)
print(ff_result.shape)

torch.Size([4, 256, 128])


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, context_length, dropout, embedding_dim):
        super().__init__()

        self.att = MultiheadAttention(num_heads=num_heads, embedding_dim=embedding_dim, context_length=context_length, dropout=dropout)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.attn_dropout = nn.Dropout(dropout)


        self.ff = FeedForward(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.ff_dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        shortcut = x #original input tensor
        x = self.layer_norm1(x) 
        x = self.att(x)
        x = self.attn_dropout(x)
        x = x + shortcut #add original i/p tensor to the output

        shortcut = x
        x = self.layer_norm2(x)
        x = self.ff(x)
        x = self.ff_dropout(x)
        x = x + shortcut

        return x


In [26]:
TB = TransformerBlock(num_heads, context_length, dropout, embedding_dim)
Trans_result = TB(IP_embedd_result)
print(result.shape)

torch.Size([4, 256, 128])


In [ ]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_length, dropout, num_heads, n_layers):
        super().__init__()

        self.input_embedding = InputEmbedding(vocab_size, embedding_dim, context_length)
        self.dropout = nn.Dropout(dropout)
        self.trans_block = nn.Sequential(*[TransformerBlock(num_heads, context_length, dropout, embedding_dim) for _ in range(n_layers)]) #stacking multiple tranformer blocks
        self.final_norm = nn.LayerNorm(embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, x):
        
        x = self.input_embedding(x)
        x = self.dropout(x)
        x = self.trans_block(x)
        x = self.final_norm(x)
        logits = self.output_layer(x)

        return logits

In [28]:
model = GPTModel(vocab_size, embedding_dim, context_length, dropout, num_heads, n_layers)
logits = model(x_batch)
print(logits.shape)

torch.Size([4, 256, 100256])


In [29]:
criterion = nn.CrossEntropyLoss()


logits_reshape = logits.view(-1, vocab_size) #B,T,V --> BxT, V
targets_reshape = y_batch.view(-1) #B, T --> N

print(logits_reshape.shape)
print(targets_reshape.shape)

loss = criterion(logits_reshape, targets_reshape)
print(f"Loss: {loss.item():.4f}")

torch.Size([1024, 100256])
torch.Size([1024])
Loss: 11.7083


In [30]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), eps=1e-08, weight_decay=0.01)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (x_batch, y_batch) in enumerate(train_dataloader):

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad() # gradients will be clearedout from the previous training
        logits = model(x_batch)
        loss = criterion(logits.view(-1, vocab_size), y_batch.view(-1))
        loss.backward() #backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #Clipping the gradients of all model parameters to a maximum norm of 1.0 to prevent exploding gradients
        optimizer.step() #update model weights and parameters
        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / len(train_dataloader) # epoch loss / len(x/y_batch)
    print(f"Epoch [{epoch+1}/{num_epochs}] | Average Loss: {avg_epoch_loss:.4f}")


Epoch [1/10] | Average Loss: 4.5423
Epoch [2/10] | Average Loss: 1.8937
Epoch [3/10] | Average Loss: 1.0700
Epoch [4/10] | Average Loss: 0.5770
Epoch [5/10] | Average Loss: 0.3181
Epoch [6/10] | Average Loss: 0.1951
Epoch [7/10] | Average Loss: 0.1351
Epoch [8/10] | Average Loss: 0.1005
Epoch [9/10] | Average Loss: 0.0794
Epoch [10/10] | Average Loss: 0.0673


In [31]:
torch.save(
    model.state_dict(),
    "gpt_verdict1.pth"
)